Sentiment Analysis using NLP Pipeline & Machine Learning


In [5]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving IMDB Dataset.csv to IMDB Dataset.csv
User uploaded file "IMDB Dataset.csv" with length 66212309 bytes


In [6]:
import pandas as pd

df = pd.read_csv('IMDB Dataset.csv')
display(df.head())

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
df.columns
df['sentiment'].value_counts()
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [10]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data
# This might take a moment if not already downloaded
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [11]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Remove HTML tags (e.g., <br />)
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphabetic characters and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize and remove stopwords and lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Apply the cleaning function to the 'review' column
clean_reviews = df['review'].apply(clean_text)

print("Original review sample:")
print(df['review'][0])
print("\nCleaned review sample:")
print(clean_reviews[0])

Original review sample:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of th

In [12]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Initialize CountVectorizer for Bag of Words
# min_df ignores words with a document frequency lower than the given threshold (e.g., 5)
# max_df ignores words with a document frequency higher than the given threshold (e.g., 0.95)
count_vectorizer = CountVectorizer(min_df=5, max_df=0.95)

# Fit and transform the cleaned reviews to get Bag of Words features
bow_features = count_vectorizer.fit_transform(clean_reviews)

print("Bag of Words feature matrix shape:", bow_features.shape)
print("Sample feature names (first 10):", count_vectorizer.get_feature_names_out()[:10])

Bag of Words feature matrix shape: (50000, 32322)
Sample feature names (first 10): ['aa' 'aaa' 'aaargh' 'aag' 'aaliyah' 'aames' 'aamir' 'aankhen' 'aapke'
 'aardman']


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dictionary to store results for easy comparison
results = []

def evaluate_model(model_name, vectorization_type, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"--- {model_name} with {vectorization_type} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\n")

    results.append({
        'Model': model_name,
        'Vectorization': vectorization_type,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })
    return accuracy

## Part A: Training and Evaluation with Bag of Words (BoW)

### Logistic Regression with BoW

In [27]:
# Logistic Regression (BoW)
log_reg_bow = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear') # Using liblinear for better convergence on sparse data
log_reg_bow.fit(X_train_bow, y_train)
y_pred_log_reg_bow = log_reg_bow.predict(X_test_bow)
evaluate_model("Logistic Regression", "BoW", y_test, y_pred_log_reg_bow)

--- Logistic Regression with BoW ---
Accuracy: 0.8884
Precision: 0.8885
Recall: 0.8884
F1 Score: 0.8884




0.8884

### Naive Bayes with BoW

In [31]:
# Naive Bayes (BoW)
# Multinomial Naive Bayes is suitable for discrete counts
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)
y_pred_mnb_bow = mnb_bow.predict(X_test_bow)
evaluate_model("Naive Bayes", "BoW", y_test, y_pred_mnb_bow)

--- Naive Bayes with BoW ---
Accuracy: 0.8582
Precision: 0.8588
Recall: 0.8582
F1 Score: 0.8581




0.8582

### Decision Tree with BoW

In [30]:
# Decision Tree (BoW)
dt_bow = DecisionTreeClassifier(random_state=42)
dt_bow.fit(X_train_bow, y_train)
y_pred_dt_bow = dt_bow.predict(X_test_bow)
evaluate_model("Decision Tree", "BoW", y_test, y_pred_dt_bow)

--- Decision Tree with BoW ---
Accuracy: 0.7275
Precision: 0.7275
Recall: 0.7275
F1 Score: 0.7275




0.7275

## Part B: Training and Evaluation with TF-IDF

### Logistic Regression with TF-IDF

In [29]:
# Logistic Regression (TF-IDF)
log_reg_tfidf = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')
log_reg_tfidf.fit(X_train_tfidf, y_train)
y_pred_log_reg_tfidf = log_reg_tfidf.predict(X_test_tfidf)
evaluate_model("Logistic Regression", "TF-IDF", y_test, y_pred_log_reg_tfidf)

--- Logistic Regression with TF-IDF ---
Accuracy: 0.8971
Precision: 0.8973
Recall: 0.8971
F1 Score: 0.8971




0.8971

### Naive Bayes with TF-IDF

In [28]:
# Naive Bayes (TF-IDF)
# Multinomial Naive Bayes also works with TF-IDF values, but they should be non-negative
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)
y_pred_mnb_tfidf = mnb_tfidf.predict(X_test_tfidf)
evaluate_model("Naive Bayes", "TF-IDF", y_test, y_pred_mnb_tfidf)

--- Naive Bayes with TF-IDF ---
Accuracy: 0.8672
Precision: 0.8673
Recall: 0.8672
F1 Score: 0.8672




0.8672

### Decision Tree with TF-IDF

In [21]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Convert 'sentiment' column to numerical labels
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

y = df['sentiment_encoded']

print("Original sentiment labels:", df['sentiment'].value_counts())
print("Encoded sentiment labels:", df['sentiment_encoded'].value_counts())
print("Mapping:", list(label_encoder.classes_), "->", label_encoder.transform(label_encoder.classes_))

Original sentiment labels: sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Encoded sentiment labels: sentiment_encoded
1    25000
0    25000
Name: count, dtype: int64
Mapping: ['negative', 'positive'] -> [0 1]


Now, we'll split both our Bag of Words (BoW) and TF-IDF features, along with the encoded sentiment labels, into training and testing sets. We'll use a standard 80/20 split, with a `random_state` for reproducibility.

In [22]:
# Split BoW features
X_train_bow, X_test_bow, y_train, y_test = train_test_split(bow_features, y, test_size=0.2, random_state=42, stratify=y)

# Split TF-IDF features. Note that y_train and y_test are consistent across both splits.
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(tfidf_features, y, test_size=0.2, random_state=42, stratify=y)

print("BoW training features shape:", X_train_bow.shape)
print("BoW testing features shape:", X_test_bow.shape)
print("TF-IDF training features shape:", X_train_tfidf.shape)
print("TF-IDF testing features shape:", X_test_tfidf.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

BoW training features shape: (40000, 32322)
BoW testing features shape: (10000, 32322)
TF-IDF training features shape: (40000, 32322)
TF-IDF testing features shape: (10000, 32322)
Training labels shape: (40000,)
Testing labels shape: (10000,)


In [23]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Convert 'sentiment' column to numerical labels
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

y = df['sentiment_encoded']

print("Original sentiment labels:", df['sentiment'].value_counts())
print("Encoded sentiment labels:", df['sentiment_encoded'].value_counts())
print("Mapping:", list(label_encoder.classes_), "->", label_encoder.transform(label_encoder.classes_))

Original sentiment labels: sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Encoded sentiment labels: sentiment_encoded
1    25000
0    25000
Name: count, dtype: int64
Mapping: ['negative', 'positive'] -> [0 1]


Now, we'll split both our Bag of Words (BoW) and TF-IDF features, along with the encoded sentiment labels, into training and testing sets. We'll use a standard 80/20 split, with a `random_state` for reproducibility.

In [24]:
# Split BoW features
X_train_bow, X_test_bow, y_train, y_test = train_test_split(bow_features, y, test_size=0.2, random_state=42, stratify=y)

# Split TF-IDF features. Note that y_train and y_test are consistent across both splits.
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(tfidf_features, y, test_size=0.2, random_state=42, stratify=y)

print("BoW training features shape:", X_train_bow.shape)
print("BoW testing features shape:", X_test_bow.shape)
print("TF-IDF training features shape:", X_train_tfidf.shape)
print("TF-IDF testing features shape:", X_test_tfidf.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

BoW training features shape: (40000, 32322)
BoW testing features shape: (10000, 32322)
TF-IDF training features shape: (40000, 32322)
TF-IDF testing features shape: (10000, 32322)
Training labels shape: (40000,)
Testing labels shape: (10000,)


In [25]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Convert 'sentiment' column to numerical labels
label_encoder = LabelEncoder()
df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

y = df['sentiment_encoded']

print("Original sentiment labels:", df['sentiment'].value_counts())
print("Encoded sentiment labels:", df['sentiment_encoded'].value_counts())
print("Mapping:", list(label_encoder.classes_), "->", label_encoder.transform(label_encoder.classes_))

Original sentiment labels: sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Encoded sentiment labels: sentiment_encoded
1    25000
0    25000
Name: count, dtype: int64
Mapping: ['negative', 'positive'] -> [0 1]


Now, we'll split both our Bag of Words (BoW) and TF-IDF features, along with the encoded sentiment labels, into training and testing sets. We'll use a standard 80/20 split, with a `random_state` for reproducibility.

In [26]:
# Split BoW features
X_train_bow, X_test_bow, y_train, y_test = train_test_split(bow_features, y, test_size=0.2, random_state=42, stratify=y)

# Split TF-IDF features. Note that y_train and y_test are consistent across both splits.
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(tfidf_features, y, test_size=0.2, random_state=42, stratify=y)

print("BoW training features shape:", X_train_bow.shape)
print("BoW testing features shape:", X_test_bow.shape)
print("TF-IDF training features shape:", X_train_tfidf.shape)
print("TF-IDF testing features shape:", X_test_tfidf.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

BoW training features shape: (40000, 32322)
BoW testing features shape: (10000, 32322)
TF-IDF training features shape: (40000, 32322)
TF-IDF testing features shape: (10000, 32322)
Training labels shape: (40000,)
Testing labels shape: (10000,)


In [32]:
# Decision Tree (TF-IDF)
dt_tfidf = DecisionTreeClassifier(random_state=42)
dt_tfidf.fit(X_train_tfidf, y_train)
y_pred_dt_tfidf = dt_tfidf.predict(X_test_tfidf)
evaluate_model("Decision Tree", "TF-IDF", y_test, y_pred_dt_tfidf)

--- Decision Tree with TF-IDF ---
Accuracy: 0.7203
Precision: 0.7204
Recall: 0.7203
F1 Score: 0.7203




0.7203

In [33]:
import pandas as pd

results_df = pd.DataFrame(results)
display(results_df.sort_values(by='F1 Score', ascending=False))

,Model,Vectorization,Accuracy,Precision,Recall,F1 Score
2,Logistic Regression,TF-IDF,0.8971,0.897333,0.8971,0.897085
0,Logistic Regression,BoW,0.8884,0.888464,0.8884,0.888395
1,Naive Bayes,TF-IDF,0.8672,0.867285,0.8672,0.867192
4,Naive Bayes,BoW,0.8582,0.858774,0.8582,0.858143
3,Decision Tree,BoW,0.7275,0.727501,0.7275,0.727500
5,Decision Tree,TF-IDF,0.7203,0.720370,0.7203,0.720278
